# EDA 4 — Frame & Boundary Effects: London-Only vs National Frame

**Purpose:** Quantify how moving from a London-only analysis to a national frame changes the cascade–counter balance and the typology — and, crucially, *decompose* that change into its two distinct causes.

### Three frames
A naïve "London vs national" comparison conflates two separate things. We separate them with an intermediate frame **B**:

| Frame | Flows included | Decile ladder | Suffix |
|-------|----------------|---------------|--------|
| **A** | London-internal only | London-relative (983 MSOAs) | `_11` / `_21` |
| **B** | London-internal only | **National** (~6,800 MSOAs) | `_intnat_` |
| **C** | London-internal **+ external** | National | `_nat_` |

- **Reclassification effect = B − A** — same flows, different decile ladder.
- **External-flow effect = C − B** — same ladder, external flows added.
- **Total = C − A** — what a naïve comparison reports (and what the earlier draft attributed entirely to external flows).

### Scope 
*(Part 0 builds the three frames; Parts 1–4 below are the analysis.)*
1. **Part 1 — Where do external flows land?** Distribution across the hierarchy, the A/B/C decomposition of the churn "boost", and a three-way *flow-direction* decomposition (within-London / outside→London / London→outside).
2. **Part 2 — What shifts in the cascade–counter balance?** Every London-vs-national comparison split into reclassification vs external.
3. **Part 3 — Does the typology change?** Typology under all three frames; switches attributed directly to reclassification (A→B) vs external flows (B→C).
4. **Part 4 — Which MSOAs are extreme?** The most cascade- and counter-dominant MSOAs, both years, both frames.

Attribute-based profiling (tenure, occupational class, prices) is **deferred to the next notebook**, which consumes the results table exported at the end here.

### Data source
`msoa_cascade_national_frame_20260623.csv` — 983 London MSOAs with London-only (`_11`/`_21`) and national-frame (`_nat_`) cascade metrics.

---


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path
from pyprojroot import here

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight'
})

# ── Paths ────────────────────────
ROOT = here()
DATA_DIR = ROOT / 'outputs'
OUTPUT_DIR = ROOT / 'outputs/eda_figs'
GEO_PATH = ROOT / 'data/london_msoa_2011.geojson'


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# ── Load data ─────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'msoa_cascade_national_frame_20260623.csv')
print(f'Loaded: {df.shape[0]} MSOAs, {df.shape[1]} columns')

# Quick check: national columns present
nat_cols = [c for c in df.columns if '_nat_' in c]
print(f'National-frame columns: {len(nat_cols)}')

Loaded: 983 MSOAs, 98 columns
National-frame columns: 36


---
# PART 0: Setup — Constructing the Three Frames

Before any analysis we assemble the three comparison frames. **0a** computes the national-frame derived metrics the preprocessing did not export; **0b** constructs **Frame B** (London-internal flows on the national ladder) — the middle term that lets Parts 1–4 separate the *reclassification* effect (B − A) from the *external-flow* effect (C − B).


## 0a. Compute Missing National-Frame Derived Metrics

The preprocessing exported base metrics and `Cascade_Dominance_nat`, but not:
- `Cross_Decile_Share_nat` — share of migration crossing decile boundaries among all migration records.
- `Sign_Concordance_nat` — whether Net_Cascade and Net_Counter share the same sign.

We compute these here using the same formulas as EDA 1.

In [3]:
# ── Cross_Decile_Share_nat ────────────────────────────────────
# Formula: (CFI_Churn + Counter_Churn) / Total_Migration
for yr in ['11', '21']:
    cfi  = df[f'CFI_Churn_nat_{yr}']
    cc   = df[f'Counter_Churn_nat_{yr}']
    tmig = df[f'Total_Migration_nat_{yr}']
    df[f'Cross_Decile_Share_nat_{yr}'] = np.where(
        tmig > 0, (cfi + cc) / tmig, 0)

# ── Sign_Concordance_nat ──────────────────────────────────────
for yr in ['11', '21']:
    nc  = df[f'Net_Cascade_nat_{yr}']
    ncc = df[f'Net_Counter_nat_{yr}']
    df[f'Sign_Concordance_nat_{yr}'] = np.where(
        (nc == 0) | (ncc == 0), 'zero',
        np.where(np.sign(nc) == np.sign(ncc), 'concordant', 'divergent')
    )

# ── Verify ────────────────────────────────────────────────────
print('=== Cross_Decile_Share_nat (2011) ===')
print(df['Cross_Decile_Share_nat_11'].describe().round(4).to_string())
print(f'\n=== Sign_Concordance_nat (2011) ===')
print(df['Sign_Concordance_nat_11'].value_counts().to_string())

print(f'\n=== Cross_Decile_Share_nat (2021) ===')
print(df['Cross_Decile_Share_nat_21'].describe().round(4).to_string())
print(f'\n=== Sign_Concordance_nat (2021) ===')
print(df['Sign_Concordance_nat_21'].value_counts().to_string())

=== Cross_Decile_Share_nat (2011) ===
count    983.0000
mean       0.8227
std        0.1031
min        0.5183
25%        0.7522
50%        0.8429
75%        0.9055
max        0.9957

=== Sign_Concordance_nat (2011) ===
Sign_Concordance_nat_11
concordant    904
divergent      78
zero            1

=== Cross_Decile_Share_nat (2021) ===
count    983.0000
mean       0.8231
std        0.1544
min        0.0000
25%        0.7805
50%        0.8588
75%        0.9148
max        0.9991

=== Sign_Concordance_nat (2021) ===
Sign_Concordance_nat_21
concordant    841
divergent     120
zero           22


### Interpretation:

- Count of 983, not 984, has been explained in data preprocess 20260622 file.

> Why the 2021 `Cross_Decile_Share_nat`(2021) has min = 0?


- The count of "Zero" of `Sign_Concordance_nat` jumped from 1 in 2011 to 22 in 2021. 
> This is consistent with "Zero" analysis in London-only version?
> - In London-only analysis, there were 2 "Zero" in 2011 and 21 in 2021. Are they the same in the national-frame analysis? If not, what have been changed?
> - Not just "Zero", other categories also changed in counts. Where are those changes in spatial distribution?
> **Check whether those 22 zero-concordance MSOAs line up with consistent 20 disclosure-suppressed MSOAs we found earlier (zero external flows in 2021) plus a couple of genuinely sparse areas, so can decide whether to flag or exclude them?**

## 0b. Construct Frame B — Internal Flows on the National Ladder

Frame A (London ladder) and Frame C (national ladder + external flows) already live in the data. Frame **B** — London-internal flows scored on the *national* ladder — is the missing middle term that lets us separate the **reclassification** effect (ladder switch) from the **external-flow** effect.

We derive B from C by removing the external contributions. For an MSOA with national decile *d* and the synthetic external node fixed at D6:
- *d* < 6 → external inflow is `Inflow_Wealthier`, external outflow is `Outflow_Wealthier`
- *d* > 6 → external inflow is `Inflow_Poorer`, external outflow is `Outflow_Poorer`
- *d* = 6 → external flows are lateral (no cascade/counter effect)

Subtracting those gives internal-only flows on the national ladder. **B's total migration must equal A's exactly** (identical internal flows) — checked below. The suffix `_intnat_` = *internal flows, national deciles*.


In [4]:
# ── Frame B: internal London flows scored on the NATIONAL ladder ──────────
EXT_DECILE = 6                       # synthetic external MSOA assigned D6

df = df.copy()

nat_dec = df['Wealth_Decile_National']

for yr in ['11', '21']:
    ext_in  = df[f'Ext_Inflow_nat_{yr}'].fillna(0)
    ext_out = df[f'Ext_Outflow_nat_{yr}'].fillna(0)

    ext_to_iw = np.where(nat_dec < EXT_DECILE, ext_in, 0)   # Inflow_Wealthier
    ext_to_ip = np.where(nat_dec > EXT_DECILE, ext_in, 0)   # Inflow_Poorer
    ext_to_ow = np.where(nat_dec < EXT_DECILE, ext_out, 0)  # Outflow_Wealthier
    ext_to_op = np.where(nat_dec > EXT_DECILE, ext_out, 0)  # Outflow_Poorer

    # B = C minus external contribution
    df[f'Inflow_Wealthier_intnat_{yr}']  = df[f'Inflow_Wealthier_nat_{yr}']  - ext_to_iw
    df[f'Outflow_Poorer_intnat_{yr}']    = df[f'Outflow_Poorer_nat_{yr}']    - ext_to_op
    df[f'Outflow_Wealthier_intnat_{yr}'] = df[f'Outflow_Wealthier_nat_{yr}'] - ext_to_ow
    df[f'Inflow_Poorer_intnat_{yr}']     = df[f'Inflow_Poorer_nat_{yr}']     - ext_to_ip

    df[f'CFI_Churn_intnat_{yr}']     = (df[f'Inflow_Wealthier_intnat_{yr}'] +
                                        df[f'Outflow_Poorer_intnat_{yr}'])
    df[f'Counter_Churn_intnat_{yr}'] = (df[f'Outflow_Wealthier_intnat_{yr}'] +
                                        df[f'Inflow_Poorer_intnat_{yr}'])
    df[f'Net_Cascade_intnat_{yr}']   = (df[f'Inflow_Wealthier_intnat_{yr}'] -
                                        df[f'Outflow_Poorer_intnat_{yr}'])
    df[f'Net_Counter_intnat_{yr}']   = (df[f'Outflow_Wealthier_intnat_{yr}'] -
                                        df[f'Inflow_Poorer_intnat_{yr}'])

    total_cross_b = df[f'CFI_Churn_intnat_{yr}'] + df[f'Counter_Churn_intnat_{yr}']
    df[f'Cascade_Dominance_intnat_{yr}'] = np.where(
        total_cross_b > 0, df[f'CFI_Churn_intnat_{yr}'] / total_cross_b, 0.5)

    df[f'Total_Migration_intnat_{yr}'] = (df[f'Total_Migration_nat_{yr}']
                                          - ext_in - ext_out)
    df[f'Cross_Decile_Share_intnat_{yr}'] = np.where(
        df[f'Total_Migration_intnat_{yr}'] > 0,
        total_cross_b / df[f'Total_Migration_intnat_{yr}'], 0)

    # Sign concordance for Frame B (same rule as A and C)
    nc, ncc = df[f'Net_Cascade_intnat_{yr}'], df[f'Net_Counter_intnat_{yr}']
    df[f'Sign_Concordance_intnat_{yr}'] = np.where(
        (nc == 0) | (ncc == 0), 'zero',
        np.where(np.sign(nc) == np.sign(ncc), 'concordant', 'divergent'))

# ── Sanity checks ─────────────────────────────────────────────────────────
print('=' * 70)
print('FRAME B SANITY CHECKS  (suffix _intnat_ = internal flows, national deciles)')
print('=' * 70)
for yr in ['11', '21']:
    diff = (df[f'Total_Migration_intnat_{yr}'] - df[f'Total_Migration_{yr}']).abs().max()
    neg = min(df[f'Inflow_Wealthier_intnat_{yr}'].min(), df[f'Outflow_Poorer_intnat_{yr}'].min(),
              df[f'Outflow_Wealthier_intnat_{yr}'].min(), df[f'Inflow_Poorer_intnat_{yr}'].min())
    print(f'\n  20{yr}:')
    print(f'    Total_Migration  B vs A  max diff : {diff:.1f}   (should be 0)')
    print(f'    Min base flow in B               : {neg:.0f}   (should be >= 0)')
    print(f'    Mean Cascade_Dominance  A={df[f"Cascade_Dominance_{yr}"].mean():.4f}  '
          f'B={df[f"Cascade_Dominance_intnat_{yr}"].mean():.4f}  '
          f'C={df[f"Cascade_Dominance_nat_{yr}"].mean():.4f}')
print('\nFrame B constructed.')


FRAME B SANITY CHECKS  (suffix _intnat_ = internal flows, national deciles)

  2011:
    Total_Migration  B vs A  max diff : 0.0   (should be 0)
    Min base flow in B               : 0   (should be >= 0)
    Mean Cascade_Dominance  A=0.4798  B=0.4815  C=0.4863

  2021:
    Total_Migration  B vs A  max diff : 0.0   (should be 0)
    Min base flow in B               : 0   (should be >= 0)
    Mean Cascade_Dominance  A=0.4638  B=0.4645  C=0.4555

Frame B constructed.


In [5]:
# ── Diagnostic: zero cross-decile-share / zero-migration MSOAs ────────────
print('=' * 70)
print('DIAGNOSTIC: zero-CDS / zero-migration MSOAs (exist identically in both frames)')
print('=' * 70)
for yr in ['11', '21']:
    nat_zero = set(df.loc[df[f'Cross_Decile_Share_nat_{yr}'] == 0, 'msoa11cd'])
    ldn_zero = set(df.loc[df[f'Cross_Decile_Share_{yr}'] == 0, 'msoa11cd'])
    print(f'\n  20{yr}:  National CDS=0: {len(nat_zero):>3d}   London CDS=0: {len(ldn_zero):>3d}'
          f'   identical set? {nat_zero == ldn_zero}')
    if nat_zero:
        m = df['msoa11cd'].isin(nat_zero)
        print(f'    all Total_Mig_nat=0? {(df.loc[m, f"Total_Migration_nat_{yr}"] == 0).all()}'
              f'   all Ext_Inflow=0? {(df.loc[m, f"Ext_Inflow_nat_{yr}"] == 0).all()}')

# ── Suppression flag: zero-migration MSOAs are disclosure artefacts ───────
# (diagnosed in EDA2/EDA3 as small-cell disclosure control, not real zeros)
for yr in ['11', '21']:
    df[f'Suppressed_{yr}'] = df[f'Total_Migration_nat_{yr}'] == 0
VALID = {yr: ~df[f'Suppressed_{yr}'] for yr in ['11', '21']}
print(f'\nSuppression flag set:  2011 -> {int(df["Suppressed_11"].sum())} MSOAs   '
      f'2021 -> {int(df["Suppressed_21"].sum())} MSOAs')
print('Policy: these are EXCLUDED from 2021 thresholds, means, typology counts and')
print("correlations, and labelled 'Suppressed' (not 'Lateral') in the typology.")
print('Valid N:  2011 =', int(VALID['11'].sum()), '  2021 =', int(VALID['21'].sum()))


DIAGNOSTIC: zero-CDS / zero-migration MSOAs (exist identically in both frames)

  2011:  National CDS=0:   0   London CDS=0:   0   identical set? True

  2021:  National CDS=0:  20   London CDS=0:  20   identical set? True
    all Total_Mig_nat=0? True   all Ext_Inflow=0? True

Suppression flag set:  2011 -> 0 MSOAs   2021 -> 20 MSOAs
Policy: these are EXCLUDED from 2021 thresholds, means, typology counts and
correlations, and labelled 'Suppressed' (not 'Lateral') in the typology.
Valid N:  2011 = 983   2021 = 963


In [6]:
# ── Extract and print the 20 suppressed MSOAs for 2021 ────────────────────
suppressed_msoas_21 = df.loc[df['Suppressed_21'], 'msoa11cd'].tolist()

print('\n' + '=' * 70)
print('LIST OF 20 SUPPRESSED MSOAs IN 2021 (For external detection)')
print('=' * 70)
print(suppressed_msoas_21)


LIST OF 20 SUPPRESSED MSOAs IN 2021 (For external detection)
['E02000053', 'E02000109', 'E02000189', 'E02000190', 'E02000220', 'E02000257', 'E02000274', 'E02000316', 'E02000365', 'E02000371', 'E02000528', 'E02000664', 'E02000726', 'E02000750', 'E02000769', 'E02000780', 'E02000809', 'E02000891', 'E02000924', 'E02006929']


We now hold all three frames in memory. Every "London vs national" comparison below is reported as **A → B → C**, so the reclassification step (B − A) and the external-flow step (C − B) are always visible separately rather than summed into a single C − A figure.

**Data-quality handling.** The diagnostic above flags MSOAs with zero recorded migration — **0 in 2011, 20 in 2021** — confirmed upstream (EDA2/EDA3) as 2021 small-cell *disclosure artefacts*, not genuine zero-movement areas. Their dominance is an imputed 0.5 (a placeholder, not a measurement), so they are excluded from all 2021 statistics and the P25 thresholds, and carry a distinct `'Suppressed'` typology label. **Reported N is 983 (2011) and 963 (2021).** The headline shift is unaffected (they contribute exactly 0 to it); the benefit is honest typology counts and an uncontaminated classification threshold.


---
# PART 1: Where Do External Flows Land?

External flows connect London MSOAs to the synthetic `EXT_OUTSIDE` MSOA (assigned national D6). Because ~69% of London MSOAs sit in national D1–D5, for those areas external inflows are `Inflow_Wealthier` and external outflows are `Outflow_Wealthier`.

This section quantifies:
- Aggregate external flow volumes (2011 vs 2021)
- How external inflows and outflows distribute across the decile hierarchy
- Which MSOAs receive the largest external flow injections

---

## 1a. Aggregate External Flow Summary

In [7]:
print('=' * 65)
print('AGGREGATE EXTERNAL FLOW SUMMARY')
print('=' * 65)

for yr in ['11', '21']:
    ei = df[f'Ext_Inflow_nat_{yr}']
    eo = df[f'Ext_Outflow_nat_{yr}']
    en = df[f'Ext_Net_nat_{yr}']
    
    print(f'\n── 20{yr} ──')
    print(f'  External inflows  (Ext→Ldn):  {ei.sum():>10,.0f} total, '
          f'{ei.mean():>6.1f} per MSOA')
    print(f'  External outflows (Ldn→Ext):  {eo.sum():>10,.0f} total, '
          f'{eo.mean():>6.1f} per MSOA')
    print(f'  Net external:                 {en.sum():>10,.0f} total, '
          f'{en.mean():>+6.1f} per MSOA')
    print(f'  London is a net {"EXPORTER" if en.sum() < 0 else "IMPORTER"} '
          f'of {abs(en.sum()):,.0f} migrants')
    print(f'  MSOAs with zero external flows: {(ei == 0).sum()}')

# Temporal change
ei_11, ei_21 = df['Ext_Inflow_nat_11'].sum(), df['Ext_Inflow_nat_21'].sum()
eo_11, eo_21 = df['Ext_Outflow_nat_11'].sum(), df['Ext_Outflow_nat_21'].sum()
print(f'\n── Temporal Change ──')
print(f'  Inflow change:  {ei_21 - ei_11:>+10,.0f}  ({(ei_21/ei_11 - 1)*100:>+.1f}%)')
print(f'  Outflow change: {eo_21 - eo_11:>+10,.0f}  ({(eo_21/eo_11 - 1)*100:>+.1f}%)')
print(f'  ⟹ External outflow nearly {eo_21/eo_11:.1f}x the 2011 level')

AGGREGATE EXTERNAL FLOW SUMMARY

── 2011 ──
  External inflows  (Ext→Ldn):     187,595 total,  190.8 per MSOA
  External outflows (Ldn→Ext):     219,221 total,  223.0 per MSOA
  Net external:                    -31,626 total,  -32.2 per MSOA
  London is a net EXPORTER of 31,626 migrants
  MSOAs with zero external flows: 0

── 2021 ──
  External inflows  (Ext→Ldn):     177,759 total,  180.8 per MSOA
  External outflows (Ldn→Ext):     353,975 total,  360.1 per MSOA
  Net external:                   -176,216 total, -179.3 per MSOA
  London is a net EXPORTER of 176,216 migrants
  MSOAs with zero external flows: 20

── Temporal Change ──
  Inflow change:      -9,836  (-5.2%)
  Outflow change:   +134,754  (+61.5%)
  ⟹ External outflow nearly 1.6x the 2011 level


### Interpretation:

**London became a more net exporter by 2021 from 2011, with less external inflows(-5.2%) and huge increase in external outflows (+61.5%).**

## 1b. How External Flows Are Classified by Decile

The synthetic external MSOA is assigned national D6. For each London MSOA, whether the external flow counts as "wealthier" or "poorer" depends on that MSOA's own national decile. This table shows the classification logic directly.

In [ ]:
EXT_DECILE = 6  # synthetic external MSOA assigned D6

print('=' * 72)
print('HOW EXTERNAL FLOWS ARE CLASSIFIED BY NATIONAL DECILE')
print('=' * 72)
print(f'\n  External MSOA decile: D{EXT_DECILE}')
print(f'  Convention: D1 = most deprived, D10 = least deprived\n')
print(f'{"Nat Decile":>10s} {"N MSOAs":>8s} {"Ext inflow is":>18s} {"Ext outflow is":>18s}')
print('-' * 58)

for d in range(1, 11):
    n = (df['Wealth_Decile_National'] == d).sum()
    if d < EXT_DECILE:
        inflow_type  = 'Inflow_Wealthier'
        outflow_type = 'Outflow_Wealthier'
    elif d > EXT_DECILE:
        inflow_type  = 'Inflow_Poorer'
        outflow_type = 'Outflow_Poorer'
    else:
        inflow_type  = 'LATERAL (same)'
        outflow_type = 'LATERAL (same)'
    print(f'  D{d:>2d}     {n:>5d}     {inflow_type:<18s} {outflow_type:<18s}')

below = (df['Wealth_Decile_National'] < EXT_DECILE).sum()
same  = (df['Wealth_Decile_National'] == EXT_DECILE).sum()
above = (df['Wealth_Decile_National'] > EXT_DECILE).sum()
print(f'\n  Summary:')
print(f'    D1–D5 (below D6): {below} MSOAs ({below/len(df)*100:.1f}%) '
      f'→ external = WEALTHIER connection')
print(f'    D6 (same):        {same} MSOAs ({same/len(df)*100:.1f}%) '
      f'→ external = LATERAL (no cascade effect)')
print(f'    D7–D10 (above):   {above} MSOAs ({above/len(df)*100:.1f}%) '
      f'→ external = POORER connection')

> **Why this table is identical for both years.** It depends only on each MSOA's national decile (fixed, from IMD 2010) and the external node's decile (fixed at D6) — neither changes across census years. Section 1b defines the *rules*; the volume flowing through each rule changes year-to-year, which 1c–1d quantify.

The majority of flows are `Outflow_Wealthier` — D1–D5 London MSOAs sending residents to the (D6-classified) external node.


## 1c. External Flow Volumes by National Decile

In [ ]:
# ── Table: external inflow & outflow by decile ────────────────
print('=' * 80)
print('EXTERNAL FLOW VOLUMES BY NATIONAL DECILE')
print('=' * 80)

dec_col = 'Wealth_Decile_National'
rows = []
for d in range(1, 11):
    mask = df[dec_col] == d
    row = {'Decile': d, 'N': mask.sum()}
    for yr in ['11', '21']:
        row[f'Ext_In_{yr}']  = df.loc[mask, f'Ext_Inflow_nat_{yr}'].sum()
        row[f'Ext_Out_{yr}'] = df.loc[mask, f'Ext_Outflow_nat_{yr}'].sum()
        row[f'Ext_Net_{yr}'] = df.loc[mask, f'Ext_Net_nat_{yr}'].sum()
    rows.append(row)

ext_dec = pd.DataFrame(rows)

print(f'\n{"Decile":>6s} {"N":>5s}  '
      f'{"Ext_In_11":>10s} {"Ext_Out_11":>10s} {"Ext_Net_11":>10s}  '
      f'{"Ext_In_21":>10s} {"Ext_Out_21":>10s} {"Ext_Net_21":>10s}')
print('-' * 80)
for _, r in ext_dec.iterrows():
    print(f'  D{r["Decile"]:>2.0f}  {r["N"]:>4.0f}  '
          f'{r["Ext_In_11"]:>10,.0f} {r["Ext_Out_11"]:>10,.0f} {r["Ext_Net_11"]:>+10,.0f}  '
          f'{r["Ext_In_21"]:>10,.0f} {r["Ext_Out_21"]:>10,.0f} {r["Ext_Net_21"]:>+10,.0f}')
print('-' * 80)
print(f'  Total {len(df):>4d}  '
      f'{ext_dec["Ext_In_11"].sum():>10,.0f} {ext_dec["Ext_Out_11"].sum():>10,.0f} '
      f'{ext_dec["Ext_Net_11"].sum():>+10,.0f}  '
      f'{ext_dec["Ext_In_21"].sum():>10,.0f} {ext_dec["Ext_Out_21"].sum():>10,.0f} '
      f'{ext_dec["Ext_Net_21"].sum():>+10,.0f}')

In [ ]:
# ── Fig 18: External flow volumes by decile ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(1, 11)
w = 0.35

# Panel (a): External inflow by decile
ax = axes[0]
ax.bar(x - w/2, ext_dec['Ext_In_11'], w, label='2011', color='#2166ac', alpha=0.7)
ax.bar(x + w/2, ext_dec['Ext_In_21'], w, label='2021', color='#b2182b', alpha=0.7)
ax.set_xlabel('National Wealth Decile')
ax.set_ylabel('Total external inflow')
ax.set_title('(a) External Inflow by Decile')
ax.set_xticks(x)
ax.set_xticklabels([f'D{d}' for d in x])
ax.axvline(5.5, color='grey', ls=':', lw=1, alpha=0.5)
ax.text(3, ax.get_ylim()[1]*0.9, '← Outside London = Wealthier areas', ha='center', fontsize=8, color='grey')
ax.text(8, ax.get_ylim()[1]*0.9, 'Outside London = Poorer areas →', ha='center', fontsize=8, color='grey')
ax.legend(fontsize=9)

# Panel (b): External outflow by decile
ax = axes[1]
ax.bar(x - w/2, ext_dec['Ext_Out_11'], w, label='2011', color='#2166ac', alpha=0.7)
ax.bar(x + w/2, ext_dec['Ext_Out_21'], w, label='2021', color='#b2182b', alpha=0.7)
ax.set_xlabel('National Wealth Decile')
ax.set_ylabel('Total external outflow')
ax.set_title('(b) External Outflow by Decile')
ax.set_xticks(x)
ax.set_xticklabels([f'D{d}' for d in x])
ax.axvline(5.5, color='grey', ls=':', lw=1, alpha=0.5)
ax.text(3, ax.get_ylim()[1]*0.9, '← Outside London = Wealthier areas', ha='center', fontsize=8, color='grey')
ax.text(8, ax.get_ylim()[1]*0.9, 'Outside London = Poorer areas →', ha='center', fontsize=8, color='grey')
ax.legend(fontsize=9)

# Panel (c): Net external by decile
ax = axes[2]
ax.bar(x - w/2, ext_dec['Ext_Net_11'], w, label='2011', color='#2166ac', alpha=0.7)
ax.bar(x + w/2, ext_dec['Ext_Net_21'], w, label='2021', color='#b2182b', alpha=0.7)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('National Wealth Decile')
ax.set_ylabel('Net external flow (In − Out)')
ax.set_title('(c) Net External Flow by Decile')
ax.set_xticks(x)
ax.set_xticklabels([f'D{d}' for d in x])
ax.legend(fontsize=9)

fig.suptitle('Fig 18: External Flow Volumes by National Wealth Decile', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_18_external_flows_by_decile.png', dpi=150)
plt.show()

### Fig 18 Interpretation

Both bar plots of external inflows and outflows are right skewed, meaning the majority of both external flows are interacted between more deprived London MSOAs and outside London areas.

But outflows from London to outside London increased significantly from 2011 to 2021, especially for London MSOAs in D2-D4. This showed more straightforward in the 3rd bar plot.

## 1d. Per-MSOA External Flow Intensity

How large are external flows relative to total migration? This determines how much leverage boundary effects have on each MSOA's cascade metrics.

In [ ]:
# ── External share of total migration ─────────────────────────
for yr in ['11', '21']:
    ext_total = df[f'Ext_Inflow_nat_{yr}'] + df[f'Ext_Outflow_nat_{yr}']
    tmig = df[f'Total_Migration_nat_{yr}']
    df[f'Ext_Share_{yr}'] = np.where(tmig > 0, ext_total / tmig * 100, 0)

print('=== External flows as % of total migration ===')
for yr in ['11', '21']:
    col = f'Ext_Share_{yr}'
    print(f'\n  20{yr}:')
    print(f'    Mean:   {df[col].mean():.1f}%')
    print(f'    Median: {df[col].median():.1f}%')
    print(f'    P90:    {df[col].quantile(0.90):.1f}%')
    print(f'    Max:    {df[col].max():.1f}%')

In [ ]:
# ── Fig 19: External share by decile (box + strip) ────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
dec_col = 'Wealth_Decile_National'

for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    col = f'Ext_Share_{yr}'
    
    data_by_dec = [df.loc[df[dec_col] == d, col].values for d in range(1, 11)]
    bp = ax.boxplot(data_by_dec, positions=range(1, 11), patch_artist=True,
                    widths=0.6, showfliers=False)
    for box in bp['boxes']:
        box.set_facecolor('#2166ac' if yr == '11' else '#b2182b')
        box.set_alpha(0.3)
    for med in bp['medians']:
        med.set_color('black')
        med.set_linewidth(2)
    
    # Overlay strip
    for d in range(1, 11):
        vals = df.loc[df[dec_col] == d, col].values
        jitter = np.random.normal(0, 0.08, size=len(vals))
        ax.scatter(d + jitter, vals, alpha=0.15, s=8,
                   color='#2166ac' if yr == '11' else '#b2182b')
    
    ax.set_xlabel('National Wealth Decile')
    ax.set_title(f'20{yr}')
    ax.set_xticks(range(1, 11))
    ax.set_xticklabels([f'D{d}' for d in range(1, 11)])

axes[0].set_ylabel('External flows as % of total migration')
fig.suptitle('Fig 19: External Flow Share by National Decile', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_19_external_share_by_decile.png', dpi=150)
plt.show()

### Fig 19 Interpretation & external flow intensity results

The last cell results showed that there is increase in external flows from 2011 to 2021.

In fig 19:
- The overall trend increased (shift upward in general) from 2011 to 2021, consistent with the increase external flows.
- Compared with more steady increase in 2011, there is a small jump between D8 and D9 in 2021.
- Both D1 and D10 look keep the unchanged during the periods.

## 1e. The "National Boost", Decomposed: Reclassification vs External Flows

The earlier version of this section computed `national − London` (C − A) and labelled the whole quantity an *external-flow* contribution. That is incorrect — C − A also contains the **reclassification** effect of switching decile ladders. With Frame B we separate them:

- **Reclassification (B − A)** — moves that change cascade/counter status purely because the ladder changed, with *identical* flows.
- **External flows (C − B)** — the extra volume injected by London↔outside moves, on the same ladder.


In [ ]:
# ── Three-way decomposition: reclassification vs external flows ──────────
print('=' * 72)
print('THREE-WAY DECOMPOSITION: Reclassification vs External Flows')
print('=' * 72)
print("""
  A = London-internal flows, London-relative deciles   (original analysis)
  B = London-internal flows, NATIONAL deciles          (same flows, new ladder)
  C = London-internal + external flows, national deciles (full national frame)

  Reclassification effect = B - A   (switching the decile ladder)
  External flow effect    = C - B   (adding external flows)
  Total difference        = C - A   (what the old comparison showed)
""")

for yr in ['11', '21']:
    print(f'\n{"-" * 72}')
    print(f'  20{yr}')
    print(f'{"-" * 72}')
    print(f'{"Metric":>22s} {"Reclass (B-A)":>14s} {"External (C-B)":>15s} '
          f'{"Total (C-A)":>13s} {"% from reclass":>15s}')
    print('-' * 82)
    for col, label in [
        ('CFI_Churn',          'Cascade churn'),
        ('Counter_Churn',      'Counter churn'),
        ('Net_Cascade',        'Net cascade'),
        ('Net_Counter',        'Net counter'),
        ('Cascade_Dominance',  'Cascade dominance'),
    ]:
        a = df[f'{col}_{yr}'].sum() if 'Dominance' not in col else df[f'{col}_{yr}'].mean()
        b = df[f'{col}_intnat_{yr}'].sum() if 'Dominance' not in col else df[f'{col}_intnat_{yr}'].mean()
        c = df[f'{col}_nat_{yr}'].sum() if 'Dominance' not in col else df[f'{col}_nat_{yr}'].mean()
        reclass, external, total = b - a, c - b, c - a
        pct = (reclass / total * 100) if abs(total) > 0.001 else 0
        if 'Dominance' in col:
            print(f'  {label:>20s}  {reclass:>+13.4f}  {external:>+14.4f}  {total:>+12.4f}  {pct:>13.1f}%')
        else:
            print(f'  {label:>20s}  {reclass:>+13,.0f}  {external:>+14,.0f}  {total:>+12,.0f}  {pct:>13.1f}%')


### Interpretation

The total "boost" the earlier analysis attributed to external flows is in fact two effects:
- **2011** — the net nudge toward cascade is only partly external; a meaningful share (~a quarter) is reclassification, because the national ladder relabels some lateral London moves as cross-decile. The old claim "external flows push cascade by +14,173" therefore overstated the external part.
- **2021** — the nudge toward counter is overwhelmingly external; reclassification is small and even slightly *opposes* it. The headline 2021 story survives, but is now *demonstrated* rather than assumed.

The per-decile picture below shows where each component bites — note especially **D6**, where external flows are lateral by construction, so any height there is *pure reclassification*.


In [ ]:
# ── Fig 20: stacked decomposition by decile ─────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
x = np.arange(1, 11); w = 0.35

for row_idx, direction in enumerate(['cascade', 'counter']):
    for col_idx, yr in enumerate(['11', '21']):
        ax = axes[row_idx, col_idx]
        reclass_by_dec, external_by_dec = [], []
        for d in range(1, 11):
            mask = df['Wealth_Decile_National'] == d
            if direction == 'cascade':
                a = (df.loc[mask, f'Inflow_Wealthier_{yr}'].sum() + df.loc[mask, f'Outflow_Poorer_{yr}'].sum())
                b = (df.loc[mask, f'Inflow_Wealthier_intnat_{yr}'].sum() + df.loc[mask, f'Outflow_Poorer_intnat_{yr}'].sum())
                c = (df.loc[mask, f'Inflow_Wealthier_nat_{yr}'].sum() + df.loc[mask, f'Outflow_Poorer_nat_{yr}'].sum())
            else:
                a = (df.loc[mask, f'Outflow_Wealthier_{yr}'].sum() + df.loc[mask, f'Inflow_Poorer_{yr}'].sum())
                b = (df.loc[mask, f'Outflow_Wealthier_intnat_{yr}'].sum() + df.loc[mask, f'Inflow_Poorer_intnat_{yr}'].sum())
                c = (df.loc[mask, f'Outflow_Wealthier_nat_{yr}'].sum() + df.loc[mask, f'Inflow_Poorer_nat_{yr}'].sum())
            reclass_by_dec.append(b - a); external_by_dec.append(c - b)

        reclass_arr = np.array(reclass_by_dec); external_arr = np.array(external_by_dec)
        color_r = '#fc8d59' if direction == 'cascade' else '#91bfdb'
        color_e = '#d7301f' if direction == 'cascade' else '#2166ac'
        ax.bar(x, reclass_arr, w * 2, label='Reclassification (B-A)', color=color_r, alpha=0.7, edgecolor='white', lw=0.5)
        ax.bar(x, external_arr, w * 2, bottom=reclass_arr, label='External flows (C-B)', color=color_e, alpha=0.7, edgecolor='white', lw=0.5)
        ax.axhline(0, color='black', lw=0.8)
        ax.set_xlabel('National Wealth Decile'); ax.set_xticks(x); ax.set_xticklabels([f'D{d}' for d in x])
        title_dir = 'Cascade' if direction == 'cascade' else 'Counter-cascade'
        ax.set_title(f'{title_dir} churn boost — 20{yr}')
        if col_idx == 0: ax.set_ylabel('Churn boost\n(persons)')
        ax.legend(fontsize=8, loc='best')

fig.suptitle('Fig 20: Decomposed Churn Boost — Reclassification vs External Flows', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'fig_20_decomposed_churn_boost.png', dpi=150); plt.show()


### Fig 20 Interpretation (decomposed)

Each bar splits the boost into **reclassification (B − A)** and **external flows (C − B)**:
- **D6** — external bars are ~zero by construction (external moves to/from D6 are lateral). Any height here is pure reclassification.
- **D1–D5** — where reclassification is large, the ladder switch itself materially changes what counts as cross-decile movement: a methodological finding in its own right, not just an external-flow artefact.
- **Temporal asymmetry** — in 2021 the external counter-cascade bars dwarf 2011 (the outflow surge), while reclassification bars stay similar across years (same flows, same ladder).


## 1f. Three-Way Flow Decomposition — Within London / Outside→London / London→Outside

The decomposition above separates *reclassification* from *external flows*. This section opens up the external flows themselves by **direction**, answering the question that most shapes the dissertation: is the counter-cascade shift driven by people **leaving** London, people **arriving** from outside, or **internal** churn?

For the ~69% of London MSOAs below D6 the directional mapping is clean:
- **Outside → London** (`Ext_Inflow`) registers as `Inflow_Wealthier` → **cascade**
- **London → Outside** (`Ext_Outflow`) registers as `Outflow_Wealthier` → **counter**
- **Within London** feeds both, per the internal flows

(For the minority above D6 the mapping flips; the code handles both.) Because the cascade/counter *labels* of external flows depend on the D6 assignment, this decomposition and the D6 choice are linked — the directional *volumes* are robust, but their cascade/counter interpretation rests on D6.


In [ ]:
# ── Decompose cross-decile churn by flow direction ──────────────────────
EXT = 6
labels = ['Within London', 'Outside->London', 'London->Outside']
store = {}
for yr in ['11', '21']:
    ei = df[f'Ext_Inflow_nat_{yr}'].fillna(0).values
    eo = df[f'Ext_Outflow_nat_{yr}'].fillna(0).values
    ndv = df['Wealth_Decile_National'].values

    out2ldn_casc = np.where(ndv < EXT, ei, 0).sum()   # Outside->London, D<6 -> Inflow_Wealthier (cascade)
    out2ldn_co   = np.where(ndv > EXT, ei, 0).sum()   # Outside->London, D>6 -> Inflow_Poorer  (counter)
    ldn2out_co   = np.where(ndv < EXT, eo, 0).sum()   # London->Outside, D<6 -> Outflow_Wealthier (counter)
    ldn2out_casc = np.where(ndv > EXT, eo, 0).sum()   # London->Outside, D>6 -> Outflow_Poorer  (cascade)

    iw_int = df[f'Inflow_Wealthier_nat_{yr}'].values  - np.where(ndv < EXT, ei, 0)
    op_int = df[f'Outflow_Poorer_nat_{yr}'].values    - np.where(ndv > EXT, eo, 0)
    ow_int = df[f'Outflow_Wealthier_nat_{yr}'].values - np.where(ndv < EXT, eo, 0)
    ip_int = df[f'Inflow_Poorer_nat_{yr}'].values     - np.where(ndv > EXT, ei, 0)
    within_casc = (iw_int + op_int).sum()
    within_co   = (ow_int + ip_int).sum()

    store[yr] = {'casc': [within_casc, out2ldn_casc, ldn2out_casc],
                 'co':   [within_co,   out2ldn_co,   ldn2out_co]}

print('=' * 84)
print('THREE-WAY FLOW DECOMPOSITION of cross-decile churn (city-wide, national frame)')
print('=' * 84)
for yr in ['11', '21']:
    cs, co = store[yr]['casc'], store[yr]['co']
    print(f'\n-- 20{yr} --')
    print(f"  {'Flow type':<20s}{'-> CASCADE':>15s}{'-> COUNTER':>15s}")
    for i, l in enumerate(labels):
        print(f'  {l:<20s}{cs[i]:>15,.0f}{co[i]:>15,.0f}')
    print(f"  {'-' * 48}")
    print(f"  {'TOTAL':<20s}{sum(cs):>15,.0f}{sum(co):>15,.0f}")
    chk_c, chk_co = df[f'CFI_Churn_nat_{yr}'].sum(), df[f'Counter_Churn_nat_{yr}'].sum()
    print(f'  reconcile: CFI_nat={chk_c:,.0f}  Counter_nat={chk_co:,.0f}  '
          f'(match: {abs(sum(cs)-chk_c) < 1 and abs(sum(co)-chk_co) < 1})')
    print(f'  aggregate dominance = {sum(cs)/(sum(cs)+sum(co)):.4f}')

delta = {}
for i, l in enumerate(labels):
    delta[l] = (store['21']['casc'][i] - store['11']['casc'][i],
                store['21']['co'][i]   - store['11']['co'][i])
print('\nWHAT DRIVES THE 2011->2021 COUNTER SHIFT?  (delta churn by flow type)')
print(f"  {'Flow type':<20s}{'d cascade':>13s}{'d counter':>13s}")
for l in labels:
    print(f'  {l:<20s}{delta[l][0]:>+13,.0f}{delta[l][1]:>+13,.0f}')

FLOW_DECOMP = {'store': store, 'delta': delta, 'labels': labels}


In [ ]:
# ── Fig 20b: cross-decile churn by flow direction, 2011 vs 2021 ─────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
xpos = np.arange(3); w = 0.38
short = ['Within\nLondon', 'Outside->\nLondon', 'London->\nOutside']

for ax, (metric, title, c11, c21) in zip(
        axes, [('casc', 'Cascade churn', '#fdae61', '#d73027'),
               ('co',   'Counter churn', '#abd9e9', '#2c7bb6')]):
    v11 = FLOW_DECOMP['store']['11'][metric]
    v21 = FLOW_DECOMP['store']['21'][metric]
    ax.bar(xpos - w/2, v11, w, label='2011', color=c11, edgecolor='white')
    ax.bar(xpos + w/2, v21, w, label='2021', color=c21, edgecolor='white')
    ax.set_xticks(xpos); ax.set_xticklabels(short, fontsize=9)
    ax.set_title(title); ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.0f}k'))
axes[0].set_ylabel('Cross-decile churn (persons)')
fig.suptitle('Fig 20b: Cross-Decile Churn by Flow Direction — 2011 vs 2021', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'fig_20b_flow_direction_decomposition.png', dpi=150); plt.show()


### Fig 20b Interpretation

The decomposition reconciles exactly with the national-frame churn totals (printed above). The temporal story is unambiguous: essentially the **entire** rise in counter-cascade churn is the **London→Outside** surge — within-London churn actually *fell* on both sides, and outside→London inflows softened. This is the quantitative signature of pandemic-era "flight from London", and it locates the counter shift in *out-migration*, not in internal restructuring.

**Caveat:** the 2021 Census was taken in March 2021 under lockdown, which both *caused* the suburbanisation and *degraded* migration measurement (more suppression/imputation). The within-London decline should not be over-read as real behavioural change.


---
# PART 2: What Shifts in the Cascade-Counter Balance?

---

## 2a. Headline Comparison: London-only vs National Frame

In [ ]:
# ── Headline: A -> B -> C decomposition (per-MSOA means) ─────────────────
print('=' * 96)
print('HEADLINE METRIC COMPARISON: A -> B -> C  (per-MSOA means)')
print('=' * 96)
metrics = [('CFI_Churn', 'Cascade churn'), ('Counter_Churn', 'Counter churn'),
           ('Net_Cascade', 'Net cascade'), ('Net_Counter', 'Net counter'),
           ('Cascade_Dominance', 'Cascade dominance'), ('Cross_Decile_Share', 'Cross-decile share')]
for yr in ['11', '21']:
    m = VALID[yr]
    print(f'\n-- 20{yr}  (N={int(m.sum())}) ' + '-' * 72)
    print(f"{'Metric':>20s}{'A (Ldn)':>13s}{'B (NatDec)':>13s}{'C (Nat+Ext)':>13s}"
          f"{'Reclass B-A':>13s}{'Ext C-B':>13s}")
    print('-' * 96)
    for col, label in metrics:
        a = df.loc[m, f'{col}_{yr}'].mean(); b = df.loc[m, f'{col}_intnat_{yr}'].mean(); c = df.loc[m, f'{col}_nat_{yr}'].mean()
        wide = ('Dominance' in col or 'Share' in col)
        fmt = '.4f' if wide else '.1f'
        print(f'  {label:>18s}{a:>13{fmt}}{b:>13{fmt}}{c:>13{fmt}}{b-a:>+13{fmt}}{c-b:>+13{fmt}}')


> **Resolving the earlier margin note** ("when we say London metrics, are they on the same decile hierarchy as national?"). **No — and that was exactly the confound.** Frame A uses the *London-relative* ladder; Frame C uses the *national* ladder. That difference is the reclassification effect (B − A), now shown separately from the external-flow effect (C − B). The "London vs national" gap was never a clean external-flow measure; the `Reclass` and `Ext` columns above split it correctly.


In [ ]:
# ── Aggregate city-wide sums (persons) ──────────────────────────────────
print('=' * 96)
print('AGGREGATE SUMS (city-wide totals, persons)')
print('=' * 96)
for yr in ['11', '21']:
    print(f'\n-- 20{yr} --')
    for m in ['CFI_Churn', 'Counter_Churn', 'Net_Cascade', 'Net_Counter']:
        a = df[f'{m}_{yr}'].sum(); b = df[f'{m}_intnat_{yr}'].sum(); c = df[f'{m}_nat_{yr}'].sum()
        print(f'  {m:>16s}:  A={a:>12,.0f}  B={b:>12,.0f}  C={c:>12,.0f}  '
              f'reclass={b-a:>+10,.0f}  external={c-b:>+10,.0f}')


## 2b. Cascade Dominance Shift, Decomposed

The per-MSOA dominance shift `national − London` (C − A) is the headline of the boundary analysis. We split it into its **reclassification** (B − A) and **external** (C − B) components, so the shift is no longer a single conflated number.


In [ ]:
# ── Decompose the per-MSOA dominance shift ──────────────────────────────
for yr in ['11', '21']:
    df[f'Dom_Reclass_{yr}']  = df[f'Cascade_Dominance_intnat_{yr}'] - df[f'Cascade_Dominance_{yr}']
    df[f'Dom_External_{yr}'] = df[f'Cascade_Dominance_nat_{yr}']    - df[f'Cascade_Dominance_intnat_{yr}']
    df[f'Dom_Shift_{yr}']    = df[f'Cascade_Dominance_nat_{yr}']    - df[f'Cascade_Dominance_{yr}']

print('=== Cascade Dominance Shift, decomposed (per-MSOA, valid only) ===')
for yr in ['11', '21']:
    m = VALID[yr]
    r, e, t = df.loc[m, f'Dom_Reclass_{yr}'], df.loc[m, f'Dom_External_{yr}'], df.loc[m, f'Dom_Shift_{yr}']
    print(f'\n  20{yr}  (N={int(m.sum())}):')
    print(f'    Reclassification (B-A): mean {r.mean():+.5f}   median {r.median():+.5f}')
    print(f'    External flows   (C-B): mean {e.mean():+.5f}   median {e.median():+.5f}')
    print(f'    Total shift      (C-A): mean {t.mean():+.5f}   median {t.median():+.5f}')
    tot = t.mean()
    if abs(tot) > 1e-9:
        print(f'    Share of mean shift:  reclass {r.mean()/tot*100:4.0f}%   external {e.mean()/tot*100:4.0f}%')
    tc = (t > 0.001).sum(); cc = (t < -0.001).sum()
    print(f'    MSOAs shifting toward cascade {tc} / counter {cc} / negligible {int(m.sum())-tc-cc}')


In [ ]:
# ── Fig 21: dominance shift by decile, decomposed (stacked) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
dec = 'Wealth_Decile_National'; x = np.arange(1, 11)
for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    r = np.nan_to_num(df.groupby(dec)[f'Dom_Reclass_{yr}'].mean().reindex(range(1, 11)).values)
    e = np.nan_to_num(df.groupby(dec)[f'Dom_External_{yr}'].mean().reindex(range(1, 11)).values)
    ax.bar(x, r, 0.7, label='Reclassification (B-A)', color='#fc8d59', alpha=0.85, edgecolor='white', lw=0.5)
    ax.bar(x, e, 0.7, bottom=r, label='External flows (C-B)', color='#2166ac', alpha=0.85, edgecolor='white', lw=0.5)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('National Wealth Decile'); ax.set_title(f'20{yr}')
    ax.set_xticks(x); ax.set_xticklabels([f'D{d}' for d in x])
    ax.text(0.02, 0.97, 'up = toward cascade', transform=ax.transAxes, fontsize=8, color='grey', va='top')
    ax.text(0.02, 0.03, 'down = toward counter', transform=ax.transAxes, fontsize=8, color='grey', va='bottom')
    ax.legend(fontsize=8, loc='lower right')
axes[0].set_ylabel('Mean dominance shift\n(decomposed)')
fig.suptitle('Fig 21: Cascade Dominance Shift by Decile — Reclassification vs External', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'fig_21_dominance_shift_decomposed.png', dpi=150); plt.show()


In [ ]:
# ── Fig 22: per-MSOA scatter — do the two effects align or oppose? ──────
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    ax.scatter(df[f'Dom_Reclass_{yr}'], df[f'Dom_External_{yr}'], s=10, alpha=0.4,
               color='#2166ac' if yr == '11' else '#b2182b', edgecolors='none')
    ax.axhline(0, color='grey', lw=0.8); ax.axvline(0, color='grey', lw=0.8)
    lim = max(df[f'Dom_Reclass_{yr}'].abs().max(), df[f'Dom_External_{yr}'].abs().max()) * 1.05
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel('Reclassification effect (B - A)'); ax.set_title(f'20{yr}')
    ax.text(0.97, 0.97, 'both -> cascade', transform=ax.transAxes, ha='right', va='top', fontsize=8, color='grey')
    ax.text(0.03, 0.03, 'both -> counter', transform=ax.transAxes, ha='left', va='bottom', fontsize=8, color='grey')
axes[0].set_ylabel('External-flow effect (C - B)')
fig.suptitle('Fig 22: Per-MSOA dominance shift — alignment of the two effects', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'fig_22_dominance_shift_scatter.png', dpi=150); plt.show()


## 2c. Cross_Decile_Share Comparison

In [ ]:
# ── Cross_Decile_Share: A -> B -> C ─────────────────────────────────────
print('=== Cross_Decile_Share: A -> B -> C ===')
for yr in ['11', '21']:
    m = VALID[yr]
    a = df.loc[m, f'Cross_Decile_Share_{yr}']; b = df.loc[m, f'Cross_Decile_Share_intnat_{yr}']; c = df.loc[m, f'Cross_Decile_Share_nat_{yr}']
    print(f'\n  20{yr}  (N={int(m.sum())}):')
    print(f'    A (Ldn)     mean {a.mean():.4f}  median {a.median():.4f}')
    print(f'    B (NatDec)  mean {b.mean():.4f}  median {b.median():.4f}   reclass {b.mean()-a.mean():+.4f}')
    print(f'    C (Nat+Ext) mean {c.mean():.4f}  median {c.median():.4f}   external {c.mean()-b.mean():+.4f}')


## 2d. Sign Concordance Comparison

In [ ]:
# ── Sign Concordance: A -> B -> C ───────────────────────────────────────
print('=== Sign Concordance: A -> B -> C ===')
for yr in ['11', '21']:
    m = VALID[yr]
    print(f'\n-- 20{yr}  (N={int(m.sum())}) --')
    for frame, suf in [('A (Ldn)', ''), ('B (NatDec)', '_intnat'), ('C (Nat+Ext)', '_nat')]:
        counts = df.loc[m, f'Sign_Concordance{suf}_{yr}'].value_counts()
        line = '   '.join(f'{v}: {int(counts.get(v, 0)):>4d}' for v in ['concordant', 'divergent', 'zero'])
        print(f'    {frame:>12s}   {line}')


---
# PART 3: Does the Typology Change?

### Methodological note (updated)

The national-frame metrics differ from London-only metrics in two ways — a **different decile ladder** and **added external flows**. The earlier draft stated these "cannot be cleanly separated in the current data." **With Frame B, they can.** We assign the typology under all three frames and attribute every switch:

- **A → B** — switches caused purely by the **decile-ladder change** (reclassification).
- **B → C** — switches caused purely by **external flows**.
- **A → C** — the total, which the cross-tabs report.

This replaces the indirect "same-decile" proxy used previously, which mis-attributed switches: a move is reclassified whenever *either* endpoint changes decile, not only when the focal MSOA does — so focal-MSOA decile stability is **not** a valid proxy for "no reclassification" (demonstrated numerically in 3c-i).

## 3a. Typology Assignment — All Three Frames

EDA 3 thresholds (`Cascade_Dominance` > 0.52 → Cascade-led; < 0.48 → Counter-led; else Symmetric; `Cross_Decile_Share` < P25 → Lateral), applied within each frame using that frame's own P25 **computed over valid (non-suppressed) MSOAs**. The 20 suppressed 2021 MSOAs carry a distinct `'Suppressed'` label rather than being miscounted as `'Lateral'`, and are excluded from the P25 so they cannot distort the classification threshold.


In [ ]:
# ── Typology under all three frames ─────────────────────────────────────
def assign_typology(dom, cds, cds_threshold, dom_upper=0.52, dom_lower=0.48):
    """Classify an MSOA into a flow-regime typology."""
    if cds < cds_threshold:
        return 'Lateral'
    if dom > dom_upper:
        return 'Cascade-led'
    if dom < dom_lower:
        return 'Counter-led'
    return 'Symmetric'

order = ['Cascade-led', 'Symmetric', 'Counter-led', 'Lateral']
order_disp = order + ['Suppressed']
frames = [('', 'A'), ('_intnat', 'B'), ('_nat', 'C')]

for yr in ['11', '21']:
    supp = df[f'Suppressed_{yr}']
    for suf, _tag in frames:
        # threshold on valid rows only, so suppressed zeros don't drag P25 down
        thr = df.loc[~supp, f'Cross_Decile_Share{suf}_{yr}'].quantile(0.25)
        typ = [assign_typology(d, c, thr)
               for d, c in zip(df[f'Cascade_Dominance{suf}_{yr}'], df[f'Cross_Decile_Share{suf}_{yr}'])]
        # suppressed MSOAs get their own label, not 'Lateral'
        df[f'Typology{suf}_{yr}'] = np.where(supp, 'Suppressed', typ)

print('=== Typology counts: A / B / C  (Suppressed shown separately) ===')
for yr in ['11', '21']:
    print(f'\n-- 20{yr}  (valid N={int((~df[f"Suppressed_{yr}"]).sum())}) --')
    print(f"{'Type':>14s}{'A (Ldn)':>11s}{'B (NatDec)':>12s}{'C (Nat+Ext)':>13s}")
    for t in order_disp:
        a = (df[f'Typology_{yr}'] == t).sum()
        b = (df[f'Typology_intnat_{yr}'] == t).sum()
        c = (df[f'Typology_nat_{yr}'] == t).sum()
        print(f'  {t:>12s}{a:>11d}{b:>12d}{c:>13d}')


## 3b. Typology Cross-Tabulation: Which MSOAs Switch?

In [ ]:
# ── Cross-tab A vs C, plus A->B / B->C / A->C switch tallies (valid only) ─
for yr in ['11', '21']:
    dv = df[VALID[yr]]
    n = len(dv)
    print(f'\n{"=" * 60}')
    print(f'TYPOLOGY CROSS-TAB: 20{yr}  (rows=A London, cols=C National, N={n})')
    print(f'{"=" * 60}')
    ct = pd.crosstab(dv[f'Typology_{yr}'], dv[f'Typology_nat_{yr}'], margins=True, margins_name='Total')
    ct = ct.reindex(index=order + ['Total'], columns=order + ['Total'], fill_value=0)
    print(ct.to_string())
    a2b = (dv[f'Typology_{yr}'] != dv[f'Typology_intnat_{yr}']).sum()
    b2c = (dv[f'Typology_intnat_{yr}'] != dv[f'Typology_nat_{yr}']).sum()
    a2c = (dv[f'Typology_{yr}'] != dv[f'Typology_nat_{yr}']).sum()
    print(f'\n  A->B (reclassification): {a2b:>3d} ({a2b/n*100:.1f}%)')
    print(f'  B->C (external flows):   {b2c:>3d} ({b2c/n*100:.1f}%)')
    print(f'  A->C (total):            {a2c:>3d} ({a2c/n*100:.1f}%)')


## 3c. Who Switches, and Why?

### 3c-i. Direct Decomposition: Reclassification (A→B) vs External Flows (B→C)

With Typology_B in hand we attribute switches directly — no proxy required — and show why the old "same-decile" split was invalid.


In [ ]:
# ── Direct switch decomposition + critique of the old proxy (valid only) ─
print('=== 3c-i  Direct switch decomposition (no proxy) ===')
for yr in ['21', '11']:
    dv = df[VALID[yr]]
    a, b, c = dv[f'Typology_{yr}'], dv[f'Typology_intnat_{yr}'], dv[f'Typology_nat_{yr}']
    a2b = (a != b); b2c = (b != c); a2c = (a != c); both = (a2b & b2c)
    print(f'\n  20{yr}  (N={len(dv)}):')
    print(f'    Pure reclassification (A->B): {a2b.sum():>3d} ({a2b.mean()*100:.1f}%)')
    print(f'    Pure external flows   (B->C): {b2c.sum():>3d} ({b2c.mean()*100:.1f}%)')
    print(f'    Total switches        (A->C): {a2c.sum():>3d} ({a2c.mean()*100:.1f}%)')
    print(f'    Affected at BOTH steps:       {both.sum():>3d}')

yr = '21'
dv = df[VALID[yr]]
same_decile = (dv['Wealth_Decile'] == dv['Wealth_Decile_National'])
dom_moves = (dv[f'Cascade_Dominance_{yr}'] != dv[f'Cascade_Dominance_intnat_{yr}'])
print(f'\n  Why the old same-decile proxy is invalid (20{yr}, N={len(dv)}):')
print(f'    MSOAs keeping the same decile number      : {same_decile.sum():>3d}')
print(f'    MSOAs whose dominance MOVES under reclass : {dom_moves.sum():>3d}')
print(f'    -> reclassification reaches far more MSOAs than the focal-decile proxy')
print(f'       implies, because a flow is reclassified whenever EITHER endpoint')
print(f'       changes decile, not only the focal MSOA. The proxy mis-attributed')
print(f'       reclassification switches to "external flows only".')


In [ ]:
# ── Switchers by decile ───────────────────────────────────────
for yr in ['21']:  # Focus on 2021 (the key period)
    switched = df[f'Typology_{yr}'] != df[f'Typology_nat_{yr}']
    df[f'Switched_{yr}'] = switched
    
    print(f'\n=== 20{yr} Switchers by National Decile ===')
    print(f'{"Decile":>8s} {"N_total":>8s} {"N_switched":>10s} {"% switched":>10s}')
    for d in range(1, 11):
        mask = df['Wealth_Decile_National'] == d
        n_total = mask.sum()
        n_sw = (mask & switched).sum()
        pct = n_sw / n_total * 100 if n_total > 0 else 0
        print(f'  D{d:>2d}    {n_total:>6d}    {n_sw:>8d}    {pct:>8.1f}%')
    
    print(f'\n=== 20{yr} Top 10 Boroughs by Switch Count ===')
    borough_switches = (df.loc[switched, 'ladnm']
                        .value_counts().head(10))
    for boro, count in borough_switches.items():
        total_in_boro = (df['ladnm'] == boro).sum()
        print(f'  {boro:>30s}: {count:>3d} / {total_in_boro} '
              f'({count/total_in_boro*100:.0f}%)')

In [ ]:
# ── Transition detail for 2021 ────────────────────────────────
yr = '21'
switched_mask = df[f'Switched_{yr}']

print(f'\n=== 20{yr} Transition Detail ===')
transitions = (df.loc[switched_mask]
               .groupby([f'Typology_{yr}', f'Typology_nat_{yr}'])
               .size().reset_index(name='count')
               .sort_values('count', ascending=False))
transitions.columns = ['From (London)', 'To (National)', 'N']
print(transitions.to_string(index=False))

## 3d. Typology Scatter: London-only vs National (2021)

In [ ]:
TYPOLOGY_COLORS = {
    'Cascade-led':  '#d6604d',
    'Symmetric':    '#66c2a5',
    'Counter-led':  '#8073ac',
    'Lateral':      '#bdbdbd'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

yr = '21'
configs = [
    (f'Cascade_Dominance_{yr}', f'Cross_Decile_Share_{yr}', 
     f'Typology_{yr}', 'London-only frame'),
    (f'Cascade_Dominance_nat_{yr}', f'Cross_Decile_Share_nat_{yr}', 
     f'Typology_nat_{yr}', 'National frame'),
]

for idx, (x_col, y_col, typ_col, title) in enumerate(configs):
    ax = axes[idx]
    for t in order:
        mask = df[typ_col] == t
        ax.scatter(df.loc[mask, x_col], df.loc[mask, y_col],
                   c=TYPOLOGY_COLORS[t], label=t, alpha=0.5, s=15, edgecolors='none')
    
    ax.axvline(0.50, color='grey', ls=':', lw=0.8, alpha=0.5)
    ax.axvline(0.52, color='black', ls='--', lw=0.8, alpha=0.5)
    ax.axvline(0.48, color='black', ls='--', lw=0.8, alpha=0.5)
    
    cds_p25 = df.loc[VALID[yr], y_col].quantile(0.25)
    ax.axhline(cds_p25, color='black', ls='--', lw=0.8, alpha=0.5)
    
    ax.set_xlabel('Cascade Dominance')
    ax.set_ylabel('Cross Decile Share')
    ax.set_title(title)
    ax.legend(fontsize=8, loc='upper right')
    
    # Count annotation
    counts = df[typ_col].value_counts()
    text = '\n'.join([f'{t}: {counts.get(t, 0)}' for t in order])
    ax.text(0.02, 0.98, text, transform=ax.transAxes, fontsize=8,
            va='top', ha='left', family='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle(f'Fig 23: Typology Scatter — London-only vs National Frame (20{yr})', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'fig_23_typology_scatter_comparison_{yr}.png', dpi=150)
plt.show()

## 3e. IMD Validation (internal coherence check)

Does the typology track subsequent IMD change? This is an *internal* credibility check, not external profiling — and it is **mildly circular**, since the deciles that define the flows are themselves IMD-based and we are predicting IMD *change*. Read it as "does the pattern cohere", not as independent validation. Genuinely external indicators (tenure, occupational class, prices) are deferred to the Phase 3 notebook.


In [ ]:
yr = '21'
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (typ_col, title) in enumerate([
    (f'Typology_{yr}', 'London-only frame'),
    (f'Typology_nat_{yr}', 'National frame'),
]):
    ax = axes[idx]
    data = [df.loc[df[typ_col] == t, 'IMD_Pctile_Change'].dropna().values
            for t in order]
    bp = ax.boxplot(data, positions=range(len(order)), patch_artist=True, widths=0.6)
    for i, box in enumerate(bp['boxes']):
        box.set_facecolor(TYPOLOGY_COLORS[order[i]])
        box.set_alpha(0.6)
    for med in bp['medians']:
        med.set_color('black')
        med.set_linewidth(2)
    
    ax.set_xticklabels(order, fontsize=9, rotation=15)
    ax.set_ylabel('IMD Pctile Change')
    ax.set_title(title)
    
    # Kruskal-Wallis
    valid_data = [g for g in data if len(g) > 0]
    if len(valid_data) > 1:
        kw_stat, kw_p = stats.kruskal(*valid_data)
        ax.annotate(f'H = {kw_stat:.1f}, p = {kw_p:.2e}',
                    xy=(0.98, 0.02), xycoords='axes fraction', ha='right', fontsize=9,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle(f'Fig 24: IMD Validation — London-only vs National Frame (20{yr})', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'fig_24_imd_validation_comparison_{yr}.png', dpi=150)
plt.show()

In [ ]:
# ── Correlation with IMD change: A / B / C  (valid only) ────────────────
yr = '21'
dv = df[VALID[yr]]
print(f'=== Correlation with IMD_Pctile_Change (20{yr}, N={len(dv)}): A / B / C ===')
print(f"{'Metric':>22s}{'A (Ldn)':>11s}{'B (NatDec)':>12s}{'C (Nat+Ext)':>13s}")
print('-' * 58)
for m in ['Cascade_Dominance', 'Cross_Decile_Share', 'Net_Cascade', 'CFI_Churn']:
    ra = dv[f'{m}_{yr}'].corr(dv['IMD_Pctile_Change'])
    rb = dv[f'{m}_intnat_{yr}'].corr(dv['IMD_Pctile_Change'])
    rc = dv[f'{m}_nat_{yr}'].corr(dv['IMD_Pctile_Change'])
    print(f'  {m:>20s}{ra:>+11.4f}{rb:>+12.4f}{rc:>+13.4f}')
print('\n(Circular by construction -- see note above. Treat as coherence, not validation.)')


---
# PART 4: Which MSOAs Are Extreme? (Phase 2)

Phase 2 of the pipeline: identify and name the most **cascade-dominant** and most **counter-dominant** MSOAs in both years. Dominance is only meaningful where cross-decile churn is non-trivial, so we restrict to non-Lateral MSOAs (CDS above the frame's P25) before ranking. We report the **national frame (C)** as primary and show each area's **London-frame (A)** dominance alongside, so frame-sensitivity is visible. These rankings — and the stability of membership across years — are the hand-off to attribute profiling.


In [ ]:
# ── Most cascade- and counter-dominant MSOAs (national frame) ───────────
TOPN = 12
print('=' * 80)
print('PHASE 2 -- MOST CASCADE- AND COUNTER-DOMINANT MSOAs (national frame)')
print('=' * 80)
extreme_ids = set()
top_casc = {}
for yr in ['11', '21']:
    dv = df[VALID[yr]]
    thr = dv[f'Cross_Decile_Share_nat_{yr}'].quantile(0.25)
    pool = dv[dv[f'Cross_Decile_Share_nat_{yr}'] >= thr]
    casc = pool.nlargest(TOPN, f'Cascade_Dominance_nat_{yr}')
    counter = pool.nsmallest(TOPN, f'Cascade_Dominance_nat_{yr}')
    top_casc[yr] = set(casc['msoa11cd'])
    extreme_ids |= set(casc['msoa11cd']) | set(counter['msoa11cd'])
    for title, sub in [('CASCADE-dominant', casc), ('COUNTER-dominant', counter)]:
        print(f'\n-- 20{yr}  Top {TOPN} {title} --')
        print(f"  {'MSOA':>11s}  {'Borough':<22s}{'Dom_C':>7s}{'Dom_A':>7s}{'CDS_C':>7s}")
        for _, r in sub.iterrows():
            print(f"  {r['msoa11cd']:>11s}  {str(r['ladnm'])[:22]:<22s}"
                  f"{r[f'Cascade_Dominance_nat_{yr}']:>7.3f}{r[f'Cascade_Dominance_{yr}']:>7.3f}"
                  f"{r[f'Cross_Decile_Share_nat_{yr}']:>7.3f}")

persist = len(top_casc['11'] & top_casc['21'])
print(f'\n  Cascade-dominant Top {TOPN}: {persist} of {TOPN} MSOAs persist across both years.')
df['Phase2_Extreme'] = df['msoa11cd'].isin(extreme_ids)
print(f"  Flagged {int(df['Phase2_Extreme'].sum())} distinct extreme MSOAs (either direction, either year).")


These named extremes — the cascade-dominant minority that bucks the city-wide counter tendency, and the most strongly counter-dominant areas — are the units the Phase 3 notebook profiles against tenure, occupational class and price indicators. The persistence figure indicates how stable "extreme" status is across the decade; low persistence would itself be a finding (extremity is episodic, not structural).


---
## Export — Results Table for the Phase 3 Notebook

The seam between this notebook and attribute profiling. We export one tidy MSOA-level row per area: identifiers, decile in both frames, `Cascade_Dominance` / `Cross_Decile_Share` / `Typology` under all three frames (both years), switch flags (A→B, B→C, A→C), the Phase-2 extreme flag, and `IMD_Pctile_Change`. **The Phase 3 notebook reads this file and joins attribute tables onto it — it never recomputes cascade logic.**


In [ ]:
# ── Export tidy results for Phase 3 ─────────────────────────────────────
keep = ['msoa11cd', 'ladnm', 'Wealth_Decile', 'Wealth_Decile_National',
        'IMD_Pctile_Change', 'Phase2_Extreme', 'Suppressed_11', 'Suppressed_21']
out = df[keep].copy()
for yr in ['11', '21']:
    for tag, suf in [('A', ''), ('B', '_intnat'), ('C', '_nat')]:
        out[f'Dom_{tag}_{yr}'] = df[f'Cascade_Dominance{suf}_{yr}']
        out[f'CDS_{tag}_{yr}'] = df[f'Cross_Decile_Share{suf}_{yr}']
        out[f'Typ_{tag}_{yr}'] = df[f'Typology{suf}_{yr}']
    out[f'switch_A2B_{yr}'] = df[f'Typology_{yr}']        != df[f'Typology_intnat_{yr}']
    out[f'switch_B2C_{yr}'] = df[f'Typology_intnat_{yr}'] != df[f'Typology_nat_{yr}']
    out[f'switch_A2C_{yr}'] = df[f'Typology_{yr}']        != df[f'Typology_nat_{yr}']

export_path = DATA_DIR / 'eda4_results_for_phase3_20260624.csv'
out.to_csv(export_path, index=False)
print(f'Exported {out.shape[0]} rows x {out.shape[1]} cols -> {export_path.name}')
print('Columns:', list(out.columns))


---
# Summary

---

In [ ]:
# ── Summary ─────────────────────────────────────────────────────────────
print('=' * 76)
print('EDA 4 -- FRAME & BOUNDARY EFFECTS: SUMMARY (Phase 1 + 2)')
print('=' * 76)

print(f'\nValid N (excl. suppressed): 2011 = {int(VALID["11"].sum())}, 2021 = {int(VALID["21"].sum())}')
print('\nPART 1-2  Decomposed cascade-dominance shift (mean per valid MSOA):')
for yr in ['11', '21']:
    m = VALID[yr]
    a = df.loc[m, f'Cascade_Dominance_{yr}'].mean(); b = df.loc[m, f'Cascade_Dominance_intnat_{yr}'].mean(); c = df.loc[m, f'Cascade_Dominance_nat_{yr}'].mean()
    rec, ext, tot = b - a, c - b, c - a
    share = f'{ext/tot*100:.0f}%' if abs(tot) > 1e-9 else 'n/a'
    print(f'  20{yr}:  A={a:.4f}  B={b:.4f}  C={c:.4f}   shift {tot:+.4f} = reclass {rec:+.4f} + external {ext:+.4f}  (external {share} of shift)')

print('\nPART 1f  Directional driver of the counter shift (delta counter churn 2011->2021):')
for l in FLOW_DECOMP['labels']:
    print(f"    {l:<20s} delta counter {FLOW_DECOMP['delta'][l][1]:>+11,.0f}")

yr = '21'; dv = df[VALID[yr]]
a2b = (dv[f'Typology_{yr}'] != dv[f'Typology_intnat_{yr}']).sum()
b2c = (dv[f'Typology_intnat_{yr}'] != dv[f'Typology_nat_{yr}']).sum()
a2c = (dv[f'Typology_{yr}'] != dv[f'Typology_nat_{yr}']).sum()
print(f'\nPART 3  Typology switches 20{yr} (N={len(dv)}):  A->C total {a2c} = reclass(A->B) {a2b} + external(B->C) {b2c} (overlap possible)')

print('\nKEY FINDINGS')
print("  1. The 'national boost' is two effects -- reclassification (ladder switch)")
print('     and external flows. In 2021 the counter nudge is overwhelmingly external.')
print('  2. That external counter push is the London->Outside out-migration surge,')
print('     not internal restructuring (within-London churn fell on both sides).')
print('  3. The counter-cascade narrative holds and sharpens once decomposed;')
print('     London-only analysis understates the 2021 counter shift.')
print('  4. Frame B makes the two effects cleanly separable, retiring the old')
print("     'cannot be separated' caveat and the invalid same-decile proxy.")
print('  5. Phase 2 extremes exported for attribute profiling in the Phase 3 notebook.')
print('  6. 20 zero-migration MSOAs (2021 disclosure artefacts) are flagged, excluded')
print("     from 2021 stats/thresholds, and labelled 'Suppressed' -- reported N: 983/963.")
